In [66]:
import requests
import matplotlib.pyplot as plt
from collections import defaultdict

# Replace with your actual league ID
league_id = "1052985975103762432"

In [67]:
import requests
import pandas as pd

# Replace with your Sleeper League ID
LEAGUE_ID = "1052985975103762432"

# Function to get player data
def get_player_data():
    url = "https://api.sleeper.app/v1/players/nfl"
    response = requests.get(url)
    players = response.json()
    player_df = pd.DataFrame.from_dict(players, orient="index")
    return player_df

# Function to get league rosters
def get_rosters(league_id):
    url = f"https://api.sleeper.app/v1/league/{league_id}/rosters"
    response = requests.get(url)
    return response.json()

# Function to get league matchups
def get_matchups(league_id, week):
    url = f"https://api.sleeper.app/v1/league/{league_id}/matchups/{week}"
    response = requests.get(url)
    return response.json()

# Function to get league users
def get_users(league_id):
    url = f"https://api.sleeper.app/v1/league/{league_id}/users"
    response = requests.get(url)
    return response.json()

# Fetch player details
player_data = get_player_data()

# Filter relevant details
player_details = player_data[["full_name", "age", "height", "years_exp", "team", "position"]]

# List of weeks (update based on your league's schedule)
weeks = range(1, 18)  # Regular season weeks

# Fetch weekly performance
performance_data = []
for week in weeks:
    matchups = get_matchups(LEAGUE_ID, week)
    for matchup in matchups:
        if "players" in matchup and matchup["players"]:
            for player_id in matchup["players"]:
                points = matchup["players_points"].get(player_id, 0)
                performance_data.append({
                    "player_id": player_id,
                    "week": week,
                    "points": points
                })

# Convert to DataFrame
performance_df = pd.DataFrame(performance_data)
performance_df = performance_df.merge(player_details, left_on="player_id", right_index=True, how="left")

# Pivot to make each week a column
performance_pivot = performance_df.pivot_table(index=["player_id", "full_name", "age", "height", "years_exp", "team", "position"], 
                                               columns="week", values="points").reset_index()
performance_pivot.columns.name = None  # Remove pivot column name

# Get league rosters and user info
rosters = get_rosters(LEAGUE_ID)
users = get_users(LEAGUE_ID)

# Mapping from roster_id to owner name
roster_owner_mapping = {roster["roster_id"]: next((user["display_name"] for user in users if user["user_id"] == roster.get("owner_id")), "Unknown") for roster in rosters}

# Build team performance DataFrame with point differential and opponents
team_data = []
for week in weeks:
    matchups = get_matchups(LEAGUE_ID, week)
    matchup_df = pd.DataFrame(matchups)
    
    # Group by matchup_id
    for matchup_id, group in matchup_df.groupby('matchup_id'):
        teams = group['roster_id'].tolist()
        points = group['points'].tolist()
        
        # For each team in the matchup
        for idx, team_id in enumerate(teams):
            team_points = points[idx]
            # Sum of opponent's points
            opponent_points = sum(points) - team_points
            point_diff = team_points - opponent_points
            
            # Get opponent team IDs (excluding current team)
            opponent_ids = [tid for tid in teams if tid != team_id]
            # Map opponent IDs to owner names
            opponent_names = [roster_owner_mapping.get(oid, "Unknown") for oid in opponent_ids]
            opponent_names_str = ', '.join(opponent_names)
            
            team_data.append({
                "team_id": team_id,
                "week": week,
                "total_points": team_points,
                "point_differential": point_diff,
                "opponent_team": opponent_names_str
            })

team_df = pd.DataFrame(team_data)

# Pivot to make each week a column
team_pivot = team_df.pivot_table(index=["team_id"], 
                                 columns="week", 
                                 values=["total_points", "point_differential", "opponent_team"], 
                                 aggfunc='first').reset_index()

# Flatten MultiIndex columns
team_pivot.columns = ['_'.join(map(str, col)).strip('_') for col in team_pivot.columns]

# Add team owner names
team_pivot["team_owner"] = team_pivot["team_id"].map(roster_owner_mapping)

# Reorder columns to move 'team_owner' next to 'team_id'
cols = team_pivot.columns.tolist()
cols.insert(1, cols.pop(cols.index('team_owner')))
team_pivot = team_pivot[cols]

# Display dataframes
print("Player Weekly Performance DataFrame (Pivoted):")
print(performance_pivot.head())

print("\nTeam Weekly Performance DataFrame (Pivoted with Point Differential and Opponent Teams):")
print(team_pivot.head())

# Save player weekly performance dataframe to CSV
performance_pivot.to_csv("player_performance.csv", index=False)

# Save team weekly performance dataframe to CSV
team_pivot.to_csv("team_performance.csv", index=False)


Player Weekly Performance DataFrame (Pivoted):
  player_id          full_name   age height  years_exp team position    1  \
0     10212         Josh Whyle  25.0     79        1.0  TEN       TE  0.0   
1     10213         Tre Tucker  23.0     69        1.0   LV       WR  3.2   
2     10214        Davis Allen  23.0     78        1.0  LAR       TE  0.0   
3     10216     Kenny McIntosh  24.0     72        1.0  SEA       RB  0.0   
4     10218  Xavier Hutchinson  24.0     75        1.0  HOU       WR  NaN   

     2     3  ...    8    9   10   11    12    13   14   15   16   17  
0  NaN   NaN  ...  4.3  1.7  0.0  1.1   0.0   0.0  0.0  0.0  0.0  0.0  
1  2.3  19.1  ...  4.3  1.5  0.0  5.0  11.7  13.4  0.0  0.0  0.0  0.0  
2  0.0   0.0  ...  0.0  1.0  5.9  0.0   0.0   0.0  0.0  0.0  0.0  0.0  
3  0.0   1.1  ...  0.0  0.0  0.0  0.0   0.0   0.0  0.0  0.0  0.0  0.0  
4  NaN   NaN  ...  NaN  0.0  1.6  0.0   0.0   0.0  0.0  0.0  0.0  0.0  

[5 rows x 24 columns]

Team Weekly Performance DataFrame 

In [68]:
# Teams and their schedules for all teams (as parsed from the image)
teams_schedule = {
    "ARI": ["@WAS", "NYG", "DAL", "SF", "CIN", "@LAR", "@SEA", "BAL", "@CLE", "ATL", "@HOU", "LAR", "@PIT", "SF", "@CHI", "@PHI", "SEA", "@ATL"],
    "ATL": ["CAR", "GB", "@DET", "@JAX", "HOU", "WAS", "@TB", "MIN", "@ARI", "BYE", "NO", "@NYJ", "TB", "@CAR", "IND", "@CHI", "@NO", "ARI"],
    "BAL": ["HOU", "@CIN", "IND", "@CLE", "@PIT", "TEN", "DET", "@ARI", "SEA", "CLE", "CIN", "BYE", "@LAC", "LAR", "@JAX", "@SF", "MIA", "PIT"],
    "BUF": ["@NYJ", "LV", "@WAS", "MIA", "JAX", "NYG", "@NE", "TB", "@CIN", "DEN", "NYJ", "BYE", "@PHI", "@KC", "DAL", "@LAC", "NE", "@MIA"],
    "CAR": ["@ATL", "NO", "@SEA", "MIN", "@DET", "@MIA", "HOU", "IND", "@CHI", "DAL", "@TEN", "@TB", "@NO", "ATL", "GB", "@JAX", "TB", "@CHI"],
    "CHI": ["GB", "@TB", "@KC", "DEN", "@WAS", "MIN", "LV", "@LAC", "@NO", "CAR", "@DET", "@MIN", "BYE", "DET", "@CLE", "ARI", "ATL", "@GB"],
    "CHI": ["TEN", "@HOU", "@IND", "LAR", "CAR", "JAX", "BYE", "@WSH", "@ARI", "NE", "GB", "MIN", "@DET", "@SF", "@MIN", "DET", "SEA", "@GB"],
    "CIN": ["NE", "@KC", "WSH", "@CAR", "BAL", "@NYG", "@CLE", "PHI", "LV", "@BAL", "@LAC", "BYE", "PIT", "@DAL", "@TEN", "CLE", "DEN", "@PIT"],
    "CLE": ["DAL", "@JAX", "NYG", "@LV", "@WSH", "@PHI", "CIN", "BAL", "LAC", "BYE", "@NO", "PIT", "@DEN", "@PIT", "KC", "@CIN", "MIA", "@BAL"],
    "DAL": ["@CLE", "NO", "BAL", "@NYG", "@PIT", "DET", "BYE", "@SF", "@ATL", "PHI", "HOU", "@WSH", "NYG", "CIN", "@CAR", "TB", "@PHI", "WSH"],
    "DEN": ["@SEA", "PIT", "@TB", "@NYJ", "LV", "LAC", "@NO", "CAR", "@BAL", "@KC", "ATL", "@LV", "CLE", "BYE", "IND", "@LAC", "@CIN", "KC"],
    "DET": ["LAR", "TB", "@ARI", "SEA", "BYE", "@DAL", "@MIN", "TEN", "@GB", "@HOU", "JAX", "@IND", "CHI", "GB", "BUF", "@CHI", "@SF", "MIN"],
    "GB": ["@PHI", "IND", "@TEN", "MIN", "@LAR", "ARI", "HOU", "@JAX", "DET", "BYE", "@CHI", "SF", "MIA", "@DET", "@SEA", "NO", "@MIN", "CHI"],
    "HOU": ["@IND", "CHI", "@MIN", "JAX", "BUF", "@NE", "@GB", "IND", "@NYJ", "DET", "@DAL", "TEN", "@JAX", "BYE", "MIA", "@KC", "BAL", "@TEN"],
    "IND": ["HOU", "@GB", "CHI", "PIT", "@JAX", "@TEN", "MIA", "@HOU", "@MIN", "BUF", "@NYJ", "DET", "@NE", "BYE", "@DEN", "TEN", "@NYG", "JAX"],
    "JAX": ["@MIA", "CLE", "@BUF", "@HOU", "IND", "@CHI", "NE", "GB", "@PHI", "MIN", "@DET", "BYE", "HOU", "@TEN", "NYJ", "@LV", "TEN", "@IND"],
    "KC": ["BAL", "CIN", "@ATL", "@LAC", "NO", "BYE", "@SF", "@LV", "TB", "DEN", "@BUF", "@CAR", "LV", "LAC", "@CLE", "HOU", "@PIT", "@DEN"],
    "LV": ["@LAC", "@BAL", "CAR", "CLE", "@DEN", "PIT", "@LAR", "KC", "@CIN", "BYE", "@MIA", "DEN", "@KC", "@TB", "ATL", "JAX", "@NO", "LAC"],
    "LAR": ["@DET", "@ARI", "SF", "@CHI", "GB", "BYE", "LV", "MIN", "@SEA", "MIA", "@NE", "PHI", "@NO", "BUF", "@SF", "@NYJ", "ARI", "SEA"],
    "LAC": ["LV", "@CAR", "@PIT", "KC", "BYE", "@DEN", "@ARI", "NO", "@CLE", "TEN", "CIN", "BAL", "@ATL", "@KC", "TB", "DEN", "@NE", "@LV"],
    "MIA": ["JAX", "BUF", "@SEA", "TEN", "@NE", "BYE", "@IND", "ARI", "@BUF", "@LAR", "LV", "NE", "@GB", "NYJ", "@HOU", "SF", "@CLE", "@NYJ"],
    "MIN": ["@NYG", "SF", "HOU", "@GB", "NYJ", "BYE", "DET", "@LAR", "IND", "@JAX", "@TEN", "@CHI", "ARI", "ATL", "CHI", "@SEA", "GB", "@DET"],
    "NE": ["@CIN", "SEA", "@NYJ", "@SF", "MIA", "HOU", "@JAX", "NYJ", "@TEN", "@CHI", "LAR", "@MIA", "IND", "BYE", "@ARI", "@BUF", "LAC", "BUF"],
    "NO": ["CAR", "@DAL", "PHI", "@ATL", "@KC", "TB", "DEN", "@LAC", "@CAR", "ATL", "CLE", "BYE", "LAR", "@NYG", "WSH", "@GB", "LV", "@TB"],
    "NYG": ["MIN", "@WSH", "@CLE", "DAL", "@SEA", "CIN", "PHI", "@PIT", "WSH", "@CAR", "BYE", "TB", "@DAL", "NO", "BAL", "@ATL", "IND", "@PHI"],
    "NYJ": ["@SF", "@TEN", "NE", "DEN", "@MIN", "BUF", "@PIT", "@NE", "HOU", "@ARI", "IND", "BYE", "SEA", "@MIA", "@JAX", "LAR", "@BUF", "MIA"],
    "PHI": ["GB", "ATL", "@NO", "@TB", "BYE", "CLE", "@NYG", "@CIN", "JAX", "@DAL", "WSH", "@LAR", "@BAL", "CAR", "PIT", "@WSH", "DAL", "NYG"],
    "PIT": ["@ATL", "@DEN", "LAC", "@IND", "DAL", "@LV", "NYJ", "NYG", "BYE", "@WSH", "BAL", "@CLE", "@CIN", "CLE", "@PHI", "@BAL", "KC", "CIN"],
    "SF": ["NYJ", "@MIN", "@LAR", "NE", "ARI", "@SEA", "KC", "DAL", "BYE", "@TB", "SEA", "@GB", "@BUF", "CHI", "LAR", "@MIA", "DET", "@ARI"],
    "SEA": ["DEN", "@NE", "MIA", "@DET", "NYG", "SF", "@ATL", "BUF", "LAR", "BYE", "@SF", "ARI", "@NYJ", "@ARI", "GB", "MIN", "@CHI", "@LAR"],
    "TB": ["WSH", "@DET", "DEN", "PHI", "@ATL", "@NO", "BAL", "ATL", "@KC", "SF", "BYE", "@NYG", "@CAR", "LV", "@LAC", "@DAL", "CAR", "NO"],
    "TEN": ["@CHI", "NYJ", "GB", "@MIA", "BYE", "IND", "@BUF", "@DET", "NE", "@LAC", "MIN", "@HOU", "@WSH", "JAX", "CIN", "@IND", "@JAX", "HOU"],
    "WSH": ["@TB", "NYG", "@CIN", "@ARI", "CLE", "@BAL", "CAR", "CHI", "@NYG", "PIT", "@PHI", "DAL", "TEN", "BYE", "@NO", "PHI", "ATL", "@DAL"],

}

# Convert the structured dictionary into a pandas DataFrame
schedule_data = {
    'week': [],
    'team': [],
    'opponent': [],
}

for team, opponents in teams_schedule.items():
    for week, opponent in enumerate(opponents, start=1):
        schedule_data['week'].append(week)
        schedule_data['team'].append(team)
        schedule_data['opponent'].append(opponent)

# Create a DataFrame from the schedule_data dictionary
schedule_df = pd.DataFrame(schedule_data)


In [70]:
# Load the player weekly performance data
performance_df = pd.read_csv("player_performance.csv")

# Ensure the 'team' column exists in performance_df
if 'team' not in performance_df.columns:
    raise ValueError("The 'team' column is missing from player_weekly_performance.csv")

# Iterate over each week to map opponents
for week in range(1, 19):  # Weeks 1 through 18
    # Filter the schedule for the current week
    week_schedule = schedule_df[schedule_df['week'] == week]
    # Create a mapping of team to opponent
    opponent_mapping = dict(zip(week_schedule['team'], week_schedule['opponent']))
    # Map the opponent to each player based on their team
    performance_df[f'opponent_week_{week}'] = performance_df['team'].map(opponent_mapping)

# Save the updated DataFrame to a new CSV file
performance_df.to_csv("player_weekly_performance_with_opponents.csv", index=False)

print("Updated 'player_weekly_performance.csv' with opponent data for each week.")


Updated 'player_weekly_performance.csv' with opponent data for each week.


In [71]:
import pandas as pd
import numpy as np

# Load the dataset
updated_performance_df = pd.read_csv("player_weekly_performance_with_opponents.csv")

# Define weeks for input and prediction
weeks = [str(i) for i in range(1, 13)]  # Weeks 1–12 as features
future_weeks = [str(i) for i in range(13, 18)]  # Weeks 13–17 to predict

# Fill missing values in the performance data
updated_performance_df[weeks] = updated_performance_df[weeks].apply(pd.to_numeric, errors='coerce').fillna(0)

# Fit a trendline and predict future weeks for each player
for index, row in updated_performance_df.iterrows():
    # Extract fantasy points (Y) for weeks 1-12 for this player
    y_values = row[weeks].values
    x_values = np.arange(1, 13)  # Week numbers (1 to 12) as X-axis values

    # Ensure Y-values are numeric
    y_values = np.array(y_values, dtype=float)

    # Fit a polynomial trendline (degree 1 for linear)
    coefficients = np.polyfit(x_values, y_values, deg=2)
    trendline = np.poly1d(coefficients)  # Construct the trendline equation

    # Predict fantasy points for weeks 13–17 using the trendline
    future_x_values = np.arange(13, 18)  # Week numbers 13–17
    predictions = trendline(future_x_values)

    # Add the predictions to the DataFrame
    for week, prediction in zip(future_weeks, predictions):
        updated_performance_df.loc[index, week] = prediction

# Save the updated DataFrame with predictions
updated_performance_df.to_csv("player_weekly_performance_with_trendline_predictions.csv", index=False)

